# 4. 对抗性攻击算法之迁移攻击（Transfer Attack）

## 4.0 上节内容回顾与本节主要内容介绍

在之前的实验中，我们所学习和使用的攻击算法均为白盒攻击算法，也就是在完全掌握模型结构与参数的前提下进行攻击。
与之相对，在不知道模型结构和参数的条件下发动攻击的算法称为黑盒攻击算法。
黑盒攻击是更贴近实际情况的攻击方式。
在本节中，我们将学习一类最常用的黑盒攻击算法：迁移攻击（Transfer Attack），并研究它的攻击效果与特点。

本实验的主要内容为采用Python、PyTorch等技术，学习并实现迁移攻击，并研究迁移攻击的性质。

## 4.1 迁移攻击介绍

对抗性样本具有迁移性，也就是说，针对一个模型产生的对抗性样本，对于其它执行相同分类任务的模型，仍然具有一定的攻击性。并且这种攻击性可以在结构不同的模型之间迁移。

利用对抗性样本的这种特性，攻击者可以在对目标模型不知情的前提下进行迁移攻击<sup>[1, 2]</sup>。
迁移攻击是一类黑盒攻击，它是指通过攻击一个替代模型来产生对抗性样本，然后用此样本去攻击目标模型，以使目标模型犯错。

参考文献：

[1] LIU Y, CHEN X, LIU C, et al. Delving into Transferable Adversarial Examples and Black­box Attacks[J/OL]. CoRR, 2016, abs/1611.02770. [arXiv:1611.02770.](http://arxiv.org/abs/1611.02770.)

[2] LI Y, BAI S, ZHOU Y, et al. Learning Transferable Adversarial Examples via Ghost Networks[J/OL]. CoRR, 2018, abs/1812.03413. [arXiv:1812.03413.](http://arxiv.org/abs/1812.03413.)

## 4.2 导入相关模块

In [1]:
import sys
sys.path.insert(0, r'D:\软件\对抗性防御\对抗性防御-1\03.代码')
import importlib.util
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import os
import logging

import test; test_fn = test.test
from loss import LabelSmoothingCrossEntropyLoss, CWLoss
from pgd import LinfPGD
from utils import load_mnist_test
from models import LeNet5, FCNet


logger = logging.getLogger('base')
logger.setLevel(logging.DEBUG)

formatter = logging.Formatter(fmt='%(process)6d %(asctime)s %(message)s', datefmt='%Y%m%d %H:%M:%S')
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.DEBUG)
stream_handler.setFormatter(formatter)

logger.addHandler(stream_handler)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 4.3 使用与目标模型结构相同的模型作为替代模型实施迁移攻击

我们首先使用与目标模型结构相同的模型作为替代模型实施迁移攻击，在此处，我们选择FCNet结构。

迁移攻击可以依靠不同的攻击算法实现，在此处我们将分别使用FGSM, PGD, CW来实现迁移攻击。

In [2]:
# 加载数据
imgs, lbls = load_mnist_test()

In [5]:
# 加载之前《对抗性攻击》课程中训练的模型，随后将测试这些模型互相进行迁移攻击的结果
fcnet_keys = ['25epoch', '50epoch', '100epoch', '200epoch']
fcnet_dict = {}

for fcnet_key in fcnet_keys:
    fcnet_state = torch.load(f'./save_model/{fcnet_key}/mnist_fcnet.pth')
    fcnet_dict[fcnet_key] = FCNet()
    fcnet_dict[fcnet_key].load_state_dict(fcnet_state['net'])
    fcnet_dict[fcnet_key] = fcnet_dict[fcnet_key].to(device)
    fcnet_dict[fcnet_key].eval()

In [6]:
# 定义攻击参数
FGSM_kwargs = dict(eps=0.1, step=1, step_size=0.1, random_start=False)
PGD_kwargs = dict(eps=0.1, step=20, step_size=0.025, random_start=True)
CW_kwargs = dict(eps=0.1, step=20, step_size=0.025, random_start=True, criterion=CWLoss)

In [9]:
# 执行干净样本测试
for k, net in fcnet_dict.items():
    cln_acc, _ = test_fn(net, imgs, lbls, bs=250)
    logger.info(f'{k} fcnet, Clean Acc: {cln_acc:.2f}')

 26588 20260207 07:42:00 25epoch fcnet, Clean Acc: 97.84
 26588 20260207 07:42:00 50epoch fcnet, Clean Acc: 98.09
 26588 20260207 07:42:00 100epoch fcnet, Clean Acc: 98.08
 26588 20260207 07:42:00 200epoch fcnet, Clean Acc: 98.09


In [10]:
attack_dict = {
    'FGSM': {'kwargs': FGSM_kwargs, 
             'result': {'sub \\ tar': [], '25epoch': [],
                        '50epoch': [], '100epoch': [], '200epoch': []}},
    'PGD': {'kwargs': PGD_kwargs, 
            'result': {'sub \\ tar': [], '25epoch': [],
                       '50epoch': [], '100epoch': [], '200epoch': []}},
    'CW': {'kwargs': CW_kwargs, 
           'result': {'sub \\ tar': [], '25epoch': [],
                      '50epoch': [], '100epoch': [], '200epoch': []}}
}
for attack_k, attack_v in attack_dict.items():
    logger.info(f'Begin {attack_k} transfer attack')
    for substitute_key, substitute_net in fcnet_dict.items():
        attack_v['result']['sub \\ tar'].append(substitute_key)

        # 更新攻击参数
        attack_v['kwargs'].update(net=substitute_net)
        transfer_attack = LinfPGD(**attack_v['kwargs'])

        for target_key, target_net in fcnet_dict.items():
            transfer_adversary = nn.Sequential(transfer_attack,  # 首先利用替代模型产生对抗性样本
                                               target_net        # 测试目标模型对迁移对抗性样本的分类结果
                                               )
            transfer_acc, _ = test_fn(transfer_adversary, imgs, lbls, bs=250, mode='attack')
            attack_v['result'][target_key].append(transfer_acc)
            logger.info(f'use {substitute_key} fcnet transfer attack {target_key} fcnet: {transfer_acc:.2f}')

 26588 20260207 07:42:00 Begin FGSM transfer attack
 26588 20260207 07:42:01 use 25epoch fcnet transfer attack 25epoch fcnet: 15.80
 26588 20260207 07:42:01 use 25epoch fcnet transfer attack 50epoch fcnet: 45.51
 26588 20260207 07:42:01 use 25epoch fcnet transfer attack 100epoch fcnet: 45.98
 26588 20260207 07:42:01 use 25epoch fcnet transfer attack 200epoch fcnet: 45.59
 26588 20260207 07:42:02 use 50epoch fcnet transfer attack 25epoch fcnet: 52.62
 26588 20260207 07:42:02 use 50epoch fcnet transfer attack 50epoch fcnet: 11.62
 26588 20260207 07:42:02 use 50epoch fcnet transfer attack 100epoch fcnet: 38.95
 26588 20260207 07:42:02 use 50epoch fcnet transfer attack 200epoch fcnet: 37.97
 26588 20260207 07:42:02 use 100epoch fcnet transfer attack 25epoch fcnet: 54.69
 26588 20260207 07:42:02 use 100epoch fcnet transfer attack 50epoch fcnet: 41.91
 26588 20260207 07:42:03 use 100epoch fcnet transfer attack 100epoch fcnet: 19.63
 26588 20260207 07:42:03 use 100epoch fcnet transfer attack 

In [12]:
"""
    此处我们使用tabulate库将实验结果以表格的形式打印，在使用之前需要`pip install tabulate`
"""
from tabulate import tabulate

for attack_k, attack_v in attack_dict.items():
    table = tabulate(attack_v['result'], headers=attack_v['result'].keys(), tablefmt='pipe', numalign='center', stralign='center')
    print(attack_k, table, '\n', sep='\n')

FGSM
|  sub \ tar  |  25epoch  |  50epoch  |  100epoch  |  200epoch  |
|:-----------:|:---------:|:---------:|:----------:|:----------:|
|   25epoch   |   15.8    |   45.51   |   45.98    |   45.59    |
|   50epoch   |   52.62   |   11.62   |   38.95    |   37.97    |
|  100epoch   |   54.69   |   41.91   |   19.63    |   38.73    |
|  200epoch   |   54.44   |   41.4    |   39.32    |   22.05    |


PGD
|  sub \ tar  |  25epoch  |  50epoch  |  100epoch  |  200epoch  |
|:-----------:|:---------:|:---------:|:----------:|:----------:|
|   25epoch   |   9.11    |   39.17   |   40.28    |   39.42    |
|   50epoch   |   49.1    |   6.11    |   33.26    |   32.72    |
|  100epoch   |   51.32   |   35.49   |    5.86    |   31.74    |
|  200epoch   |   50.88   |   33.63   |   31.06    |    5.81    |


CW
|  sub \ tar  |  25epoch  |  50epoch  |  100epoch  |  200epoch  |
|:-----------:|:---------:|:---------:|:----------:|:----------:|
|   25epoch   |   10.11   |   82.18   |   82.63    |   81.59

从上面的结果可以看出，基于FGSM和PGD的迁移攻击具有一定的攻击性，但一般弱于白盒攻击（对角线上的结果）。基于CW攻击产生的对抗性样本几乎没有迁移性。

## 4.4 使用与目标模型结构不同的模型作为替代模型实施迁移攻击

在4.3中，我们测试了相同结构模型之间的迁移攻击效果，在本小节，我们将测试不同结构的模型之间迁移攻击的效果。

In [13]:
# 加载模型
# 分别加载FCNet和LeNet5的模型
diff_net_key_module = {'200epoch/mnist_fcnet': FCNet, '50epoch/mnist_lenet5': LeNet5}
diff_net_dict = {}

for k, m in diff_net_key_module.items():
    state = torch.load(f'./save_model/{k}.pth')
    diff_net_dict[k] = m()
    diff_net_dict[k].load_state_dict(state['net'])
    diff_net_dict[k] = diff_net_dict[k].to(device)
    diff_net_dict[k].eval()

In [27]:
# 执行干净样本测试
for k, net in diff_net_dict.items():
    acc, _ = test_fn(net, imgs, lbls, bs=250)
    logger.info(f'{k} Clean Acc: {acc:.2f}')

 26588 20260207 07:59:19 200epoch/mnist_fcnet Clean Acc: 98.09
 26588 20260207 07:59:19 50epoch/mnist_lenet5 Clean Acc: 99.00


In [28]:
attack_dict_diff = {
    'FGSM': {'kwargs': FGSM_kwargs, 
             'result': {'sub \\ tar': [], '200epoch/mnist_fcnet': [], '50epoch/mnist_lenet5': []}},
    'PGD': {'kwargs': PGD_kwargs, 
            'result': {'sub \\ tar': [], '200epoch/mnist_fcnet': [], '50epoch/mnist_lenet5': []}},
    'CW': {'kwargs': CW_kwargs, 
           'result': {'sub \\ tar': [], '200epoch/mnist_fcnet': [], '50epoch/mnist_lenet5': []}}
}
for attack_k, attack_v in attack_dict_diff.items():
    logger.info(f'Begin {attack_k} transfer attack')
    for substitute_key, substitute_net in diff_net_dict.items():
        attack_v['result']['sub \\ tar'].append(substitute_key)

        # 更新攻击参数
        attack_v['kwargs'].update(net=substitute_net)
        transfer_attack = LinfPGD(**attack_v['kwargs'])

        for target_key, target_net in diff_net_dict.items():
            transfer_adversary = nn.Sequential(transfer_attack,  # 首先利用替代模型产生对抗性样本
                                               target_net        # 测试目标模型对迁移对抗性样本的分类结果
                                               )
            transfer_acc, _ = test_fn(transfer_adversary, imgs, lbls, bs=250, mode='attack')
            attack_v['result'][target_key].append(transfer_acc)
            logger.info(f'use {substitute_key} fcnet transfer attack {target_key} fcnet: {transfer_acc:.2f}')

 26588 20260207 07:59:19 Begin FGSM transfer attack
 26588 20260207 07:59:20 use 200epoch/mnist_fcnet fcnet transfer attack 200epoch/mnist_fcnet fcnet: 22.05
 26588 20260207 07:59:20 use 200epoch/mnist_fcnet fcnet transfer attack 50epoch/mnist_lenet5 fcnet: 97.35
 26588 20260207 07:59:20 use 50epoch/mnist_lenet5 fcnet transfer attack 200epoch/mnist_fcnet fcnet: 95.55
 26588 20260207 07:59:21 use 50epoch/mnist_lenet5 fcnet transfer attack 50epoch/mnist_lenet5 fcnet: 79.45
 26588 20260207 07:59:21 Begin PGD transfer attack
 26588 20260207 07:59:22 use 200epoch/mnist_fcnet fcnet transfer attack 200epoch/mnist_fcnet fcnet: 5.84
 26588 20260207 07:59:24 use 200epoch/mnist_fcnet fcnet transfer attack 50epoch/mnist_lenet5 fcnet: 97.40
 26588 20260207 07:59:28 use 50epoch/mnist_lenet5 fcnet transfer attack 200epoch/mnist_fcnet fcnet: 95.08
 26588 20260207 07:59:32 use 50epoch/mnist_lenet5 fcnet transfer attack 50epoch/mnist_lenet5 fcnet: 33.72
 26588 20260207 07:59:32 Begin CW transfer attack


In [29]:
for attack_k, attack_v in attack_dict_diff.items():
    table = tabulate(attack_v['result'], headers=attack_v['result'].keys(), tablefmt='pipe', numalign='center', stralign='center')
    print(attack_k, table, '\n', sep='\n')

FGSM
|      sub \ tar       |  200epoch/mnist_fcnet  |  50epoch/mnist_lenet5  |
|:--------------------:|:----------------------:|:----------------------:|
| 200epoch/mnist_fcnet |         22.05          |         97.35          |
| 50epoch/mnist_lenet5 |         95.55          |         79.45          |


PGD
|      sub \ tar       |  200epoch/mnist_fcnet  |  50epoch/mnist_lenet5  |
|:--------------------:|:----------------------:|:----------------------:|
| 200epoch/mnist_fcnet |          5.84          |          97.4          |
| 50epoch/mnist_lenet5 |         95.08          |         33.72          |


CW
|      sub \ tar       |  200epoch/mnist_fcnet  |  50epoch/mnist_lenet5  |
|:--------------------:|:----------------------:|:----------------------:|
| 200epoch/mnist_fcnet |          6.11          |         98.75          |
| 50epoch/mnist_lenet5 |         96.98          |          33.4          |




可以看到，FCNet与LeNet5之间很难互相迁移攻击成功。然而，一些文献指出即使是结构完全不同的模型也能够互相迁移攻击成功。那么，是什么导致了FCNet与LeNet5之间迁移攻击失败呢？大家可以思考一下。

其实，对抗性样本的迁移性来自于不同的模型在训练时，从训练数据中学习到相同或者相似的特征。当攻击者利用替代模型修改样本的特征，使网络识别其为错误类别时，目标模型大概率也会被修改后的特征欺骗。然而，FCNet与LeNet5之间存在一个较大的差别，FCNet会将输入的图像重塑成一个向量，而LeNet5则利用卷积层直接提取图像上的特征。这一差别将使得FCNet与LeNet5学习到的特征具有较大的差别，因此它们之间难以迁移攻击成功。

## 4.5 研究Label Smoothing模型能否防御迁移攻击

In [30]:
# 模型加载
# Label Smoothing模型，使用之前表现最好的平滑系数为0.7的模型
ls_state = torch.load('./save_model/50epoch/mnist_lenet5_ls0.7.pth')
ls_lenet = LeNet5()
ls_lenet.load_state_dict(ls_state['net'])
ls_lenet = ls_lenet.to(device)
ls_lenet.eval()

# 标准模型
std_state = torch.load('./save_model/50epoch/mnist_lenet5.pth')
std_lenet = LeNet5()
std_lenet.load_state_dict(std_state['net'])
std_lenet = std_lenet.to(device)
std_lenet.eval()

LeNet5(
  (conv1): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Sequential(
    (0): Linear(in_features=400, out_features=120, bias=True)
    (1): ReLU()
  )
  (fc2): Sequential(
    (0): Linear(in_features=120, out_features=84, bias=True)
    (1): ReLU()
  )
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [31]:
# 定义攻击参数
FGSM_kwargs = dict(eps=0.1, step=1, step_size=0.1, random_start=False)
PGD_kwargs = dict(eps=0.1, step=20, step_size=0.025, random_start=True)
CW_kwargs = dict(eps=0.1, step=20, step_size=0.025, random_start=True, criterion=CWLoss)

In [33]:
# 以标准模型作为替代模型攻击LS模型
FGSM_kwargs.update(net=std_lenet)
PGD_kwargs.update(net=std_lenet)
CW_kwargs.update(net=std_lenet)

# 创建攻击
fgsm_std_transfer_adversary = LinfPGD(**FGSM_kwargs)
pgd_std_transfer_adversary = LinfPGD(**PGD_kwargs)
cw_std_transfer_adversary = LinfPGD(**CW_kwargs)

# 执行迁移攻击
trans_fgsm_acc, _ = test_fn(nn.Sequential(fgsm_std_transfer_adversary, ls_lenet), imgs, lbls, bs=250, mode='attack')
logger.info(f'以标准模型作为替代模型，用FGSM迁移攻击LS模型 Acc: {trans_fgsm_acc:.2f}')

trans_pgd_acc, _ = test_fn(nn.Sequential(pgd_std_transfer_adversary, ls_lenet), imgs, lbls, bs=250, mode='attack')
logger.info(f'以标准模型作为替代模型，用PGD20迁移攻击LS模型 Acc: {trans_pgd_acc:.2f}')

trans_cw_acc, _ = test_fn(nn.Sequential(cw_std_transfer_adversary, ls_lenet), imgs, lbls, bs=250, mode='attack')
logger.info(f'以标准模型作为替代模型，用CW20迁移攻击LS模型 Acc: {trans_cw_acc:.2f}')

 26588 20260207 08:00:57 以标准模型作为替代模型，用FGSM迁移攻击LS模型 Acc: 96.97
 26588 20260207 08:01:00 以标准模型作为替代模型，用PGD20迁移攻击LS模型 Acc: 96.49
 26588 20260207 08:01:07 以标准模型作为替代模型，用CW20迁移攻击LS模型 Acc: 98.55


可以看到，Label Smoothing模型能够很好地防御以标准训练模型作为替代模型的迁移攻击。